In [2]:

import requests
from bs4 import BeautifulSoup
import pandas as pd

# Website URL
url = "https://books.toscrape.com/"

# Website se data fetch karna
response = requests.get(url)

# Check website connection
print("Status Code:", response.status_code)

# HTML parse karna
soup = BeautifulSoup(response.text, "html.parser")

# Website title
print("Website Title:", soup.title.text)

# First book details
first_book = soup.find("article", class_="product_pod")

book_name = first_book.h3.a["title"]
book_price = first_book.find("p", class_="price_color").text
availability = first_book.find("p", class_="instock availability").text.strip()

print("\nFirst Book Details")
print("Book Name:", book_name)
print("Price:", book_price)
print("Availability:", availability)


Status Code: 200
Website Title: 
    All products | Books to Scrape - Sandbox


First Book Details
Book Name: A Light in the Attic
Price: Â£51.77
Availability: In stock


In [3]:

# Fix currency encoding
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

first_book = soup.find("article", class_="product_pod")

print("Book:", first_book.h3.a["title"])
print("Price:", first_book.find("p", class_="price_color").text)


Book: A Light in the Attic
Price: £51.77


In [5]:

from urllib.parse import urljoin

books = []

book_cards = soup.find_all("article", class_="product_pod")

for book in book_cards:

    name = book.h3.a["title"]

    price = book.find(
        "p", class_="price_color"
    ).text.strip()

    availability = book.find(
        "p", class_="instock availability"
    ).text.strip()

    rating = book.find(
        "p", class_="star-rating"
    )["class"][1]

    book_url = urljoin(
        url, book.h3.a["href"]
    )

    books.append({
        "Book Name": name,
        "Price": price,
        "Rating": rating,
        "Availability": availability,
        "Product URL": book_url
    })

# Convert into DataFrame
df = pd.DataFrame(books)

print("Total Books:", len(df))

df.head(10)


Total Books: 20


,Book Name,Price,Rating,Availability,Product URL
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...
5,The Requiem Red,£22.65,One,In stock,https://books.toscrape.com/catalogue/the-requi...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four,In stock,https://books.toscrape.com/catalogue/the-dirty...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three,In stock,https://books.toscrape.com/catalogue/the-comin...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four,In stock,https://books.toscrape.com/catalogue/the-boys-...
9,The Black Maria,£52.15,One,In stock,https://books.toscrape.com/catalogue/the-black...


In [6]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

base_url = "https://books.toscrape.com/catalogue/"
next_page = "https://books.toscrape.com/"

all_books = []

while next_page:

    response = requests.get(next_page)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    book_cards = soup.find_all(
        "article", class_="product_pod"
    )

    for book in book_cards:

        name = book.h3.a["title"]

        price = book.find(
            "p", class_="price_color"
        ).text.strip()

        availability = book.find(
            "p", class_="instock availability"
        ).text.strip()

        rating = book.find(
            "p", class_="star-rating"
        )["class"][1]

        product_url = urljoin(
            next_page, book.h3.a["href"]
        )

        all_books.append({
            "Book Name": name,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Product URL": product_url
        })

    next_button = soup.find("li", class_="next")

    if next_button:
        next_page = urljoin(
            next_page,
            next_button.a["href"]
        )
    else:
        next_page = None

df = pd.DataFrame(all_books)

print("Total Books Scraped:", len(df))
print("Total Columns:", len(df.columns))

df.head()


Total Books Scraped: 1000
Total Columns: 5


,Book Name,Price,Rating,Availability,Product URL
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [7]:

df.to_csv(
    "CodeAlpha_Web_Scraping.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV file successfully saved!")



CSV file successfully saved!


In [8]:
from google.colab import files
files.download("CodeAlpha_Web_Scraping.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:

# Create a copy for cleaning
df_clean = df.copy()

# Convert price into numeric values
df_clean["Price"] = (
    df_clean["Price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

# Convert rating into numeric values
rating_mapping = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_clean["Rating"] = df_clean["Rating"].map(rating_mapping)

# Check dataset
print("Dataset Shape:", df_clean.shape)
print("\nData Types:")
print(df_clean.dtypes)

print("\nAverage Book Price:", df_clean["Price"].mean())
print("Highest Book Price:", df_clean["Price"].max())
print("Lowest Book Price:", df_clean["Price"].min())

# Save final dataset
df_clean.to_csv(
    "CodeAlpha_Web_Scraping_Final.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nFinal Dataset Saved Successfully!")


Dataset Shape: (1000, 5)

Data Types:
Book Name        object
Price           float64
Rating            int64
Availability     object
Product URL      object
dtype: object

Average Book Price: 35.07035
Highest Book Price: 59.99
Lowest Book Price: 10.0

Final Dataset Saved Successfully!
